# 풍투데이 크롤링 취합본

#### - 260420_crawling.ipynb: 풍투데이 랭킹 기준 수집
#### - 260423_crawling.ipynb: 소프트콘 SOOP 기준 추가 수집 + 통합

In [1]:
# ============================================================
import os
import time
from typing import Iterable, Optional, Tuple

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

### 0단계 : 기본 설정

In [2]:
# ============================================================
BASE_URL = "https://poong.today"
RANKING_URL = f"{BASE_URL}/rankings/broadcast"

START_YEAR = 2025
START_MONTH = 1
END_YEAR = 2026
END_MONTH = 3

SAVE_ROOT = "poong_save"
NULL_DIR = os.path.join(SAVE_ROOT, "channel_id_null")
WRONG_DIR = os.path.join(SAVE_ROOT, "channel_id_wrong")
DONE_DIR = os.path.join(SAVE_ROOT, "done")
RESULT_DIR = os.path.join(SAVE_ROOT, "result")

for folder in [NULL_DIR, WRONG_DIR, DONE_DIR, RESULT_DIR]:
    os.makedirs(folder, exist_ok=True)


def make_driver(headless: bool = False) -> webdriver.Chrome:
    """크롬 드라이버 생성."""
    options = Options()
    options.add_argument("start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    if headless:
        options.add_argument("--headless=new")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def clean_channel_id(series: pd.Series) -> pd.Series:
    """channel_id를 문자열로 정리."""
    return series.astype(str).str.strip()


def is_missing_channel_id(series: pd.Series) -> pd.Series:
    """channel_id 결측/문자형 결측 판별."""
    return series.isna() | series.astype(str).str.strip().isin(["", "None", "nan", "NaN"])


def month_in_range(year: int, month: int) -> bool:
    """2025.01 ~ 2026.03 범위 여부."""
    return (year == 2025 and month >= 1) or (year == 2026 and month <= 3)


def parse_star_value(text: Optional[str]) -> int:
    """'1,234' 같은 문자열을 정수로 변환."""
    if text is None:
        return 0
    text = str(text).replace(",", "").strip()
    return int(text) if text.isdigit() else 0


def extract_monthly_star_from_page(driver: webdriver.Chrome) -> Tuple[int, int, int, int]:
    """현재 풍투데이 채널 상세 페이지에서 연/월/누적별풍/시급풍 추출."""
    year = int(driver.find_element(By.CLASS_NAME, "small-year").text.strip())
    month = int(driver.find_element(By.CLASS_NAME, "small-month").text.strip())

    total_star = 0
    hourly_star = 0

    lis = driver.find_elements(By.TAG_NAME, "li")
    for li in lis:
        text = li.text
        if "누적 별풍선" in text:
            total_star = parse_star_value(li.find_element(By.TAG_NAME, "span").text)
        elif "시급(풍)" in text:
            hourly_star = parse_star_value(li.find_element(By.TAG_NAME, "span").text)

    return year, month, total_star, hourly_star

### 1단계 : 풍투데이 랭킹에서 버추얼 스트리머 channel_id 수집

In [3]:
# ============================================================
def crawl_virtual_ranking(
    start_rank: int = 1,
    end_rank: int = 100,
    output_file: str = "poongtoday_vtuber_ranking_ids.csv",
    headless: bool = False,
) -> pd.DataFrame:
    """
    풍투데이 방송 랭킹 페이지에서 '버추얼' + '지난 달' 기준 랭킹 ID 수집.
    예: start_rank=1, end_rank=100 또는 101~400, 401~700처럼 구간 수집 가능.
    """
    driver = make_driver(headless=headless)
    wait = WebDriverWait(driver, 15)
    data = []

    try:
        driver.get(RANKING_URL)

        dropdown = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".react-dropdown-select")))
        driver.execute_script("arguments[0].click();", dropdown)

        virtual_option = wait.until(
            EC.presence_of_element_located((By.XPATH, "//span[@role='option' and contains(text(),'버추얼')]"))
        )
        driver.execute_script("arguments[0].click();", virtual_option)
        time.sleep(2)

        last_month = wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(text(),'지난 달')]")))
        driver.execute_script("arguments[0].click();", last_month)
        time.sleep(3)

        while True:
            rows = driver.find_elements(By.CSS_SELECTOR, "li.row")
            if len(rows) >= end_rank:
                print(f"{len(rows)}명의 데이터가 로드되었습니다. 추출을 시작합니다.")
                break

            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1)
            try:
                more_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//button[contains(text(),'더보기')]")))
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(2)
            except TimeoutException:
                print("더 이상 '더보기' 버튼을 찾을 수 없습니다.")
                break

        rows = driver.find_elements(By.CSS_SELECTOR, "li.row")
        for row in rows:
            try:
                rank_text = row.find_element(By.CSS_SELECTOR, ".rank").text
                if not rank_text.isdigit():
                    continue

                rank = int(rank_text)
                if start_rank <= rank <= end_rank:
                    name = row.find_element(By.CSS_SELECTOR, ".nick").text.split("\n")[0]
                    link = row.find_element(By.CSS_SELECTOR, ".subject a").get_attribute("href")
                    channel_id = link.split("/")[-1]
                    data.append([rank, name, channel_id])
                    print(f"수집 완료: {rank}위 - {name} ({channel_id})")
                elif rank > end_rank:
                    break
            except Exception:
                continue

    finally:
        driver.quit()

    df_rank = pd.DataFrame(data, columns=["rank", "name", "channel_id"])
    df_rank.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"랭킹 ID 저장 완료: {output_file} / {len(df_rank)}명")
    return df_rank

In [ ]:
# ============================================================
# 2. channel_id 목록의 월별 별풍/시급풍 수집

### 2단계 : 그 channel_id로 풍투데이 상세페이지 들어가서 2025.01 ~ 2026.03 월별 별풍/시급풍 수집

In [4]:
# ============================================================
def crawl_monthly_poong_by_ids(
    target_df: pd.DataFrame,
    output_file: str,
    wrong_file: Optional[str] = None,
    rank_col: str = "rank",
    name_col: str = "name",
    channel_col: str = "channel_id",
    max_clicks: int = 20,
    headless: bool = False,
    append_each_streamer: bool = True,
) -> pd.DataFrame:
    """
    channel_id별 2025.01~2026.03 월별 누적별풍/시급풍 수집.
    append_each_streamer=True면 한 명 수집이 끝날 때마다 바로 CSV에 이어쓰기.
    중간에 꺼져도 저장된 output_file/wrong_file 기준으로 이어서 실행 가능.
    """
    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    if wrong_file:
        os.makedirs(os.path.dirname(wrong_file) or ".", exist_ok=True)

    target_df = target_df.copy()
    target_df[channel_col] = clean_channel_id(target_df[channel_col])

    existing_ids = set()
    for path in [output_file, wrong_file]:
        if path and os.path.exists(path):
            try:
                checked = pd.read_csv(path)
                if channel_col in checked.columns:
                    existing_ids.update(clean_channel_id(checked[channel_col]).dropna().unique())
            except Exception:
                pass

    target_df = target_df[~target_df[channel_col].isin(existing_ids)].drop_duplicates(channel_col)
    print(f"새롭게 수집할 타겟: {len(target_df)}명")

    if target_df.empty:
        return pd.DataFrame()

    driver = make_driver(headless=headless)
    session_results = []

    try:
        for _, row in target_df.iterrows():
            channel_id = str(row[channel_col]).strip()
            streamer_name = row.get(name_col, "Unknown")
            rank = row.get(rank_col, None)
            print(f"[{streamer_name} ({channel_id})] 수집 시작...")

            driver.get(f"{BASE_URL}/broadcast/{channel_id}")
            time.sleep(2)

            temp_results = []
            has_data = False

            try:
                for _ in range(max_clicks):
                    time.sleep(1.5)
                    year, month, total_star, hourly_star = extract_monthly_star_from_page(driver)

                    if month_in_range(year, month):
                        item = {
                            "rank": rank,
                            "name": streamer_name,
                            "channel_id": channel_id,
                            "year": year,
                            "month": month,
                            "total_star": total_star,
                            "hourly_star": hourly_star,
                        }
                        temp_results.append(item)
                        has_data = True
                        print(f"   - {year}-{month:02d} 수집 완료")

                    elif year < START_YEAR:
                        print("   - 2024년 이전 데이터 도달. 다음 채널로 이동합니다.")
                        break

                    driver.find_element(By.CSS_SELECTOR, ".small-month .left-arrow").click()

            except NoSuchElementException:
                print("   - 더 이상의 과거 기록이 없습니다.")
            except Exception as e:
                print(f"   - 돌발 에러 발생: {e}")
                raise

            if has_data:
                df_temp = pd.DataFrame(temp_results)
                session_results.extend(temp_results)
                if append_each_streamer:
                    header = not os.path.exists(output_file)
                    df_temp.to_csv(output_file, mode="a", index=False, header=header, encoding="utf-8-sig")
            else:
                print(f"   ⚠️ {streamer_name}: 수집된 기록 없음")
                if wrong_file:
                    df_wrong = pd.DataFrame([{"name": streamer_name, "channel_id": channel_id}])
                    header = not os.path.exists(wrong_file)
                    df_wrong.to_csv(wrong_file, mode="a", index=False, header=header, encoding="utf-8-sig")

            print("-" * 30)

    finally:
        driver.quit()
        print("이번 크롤링 세션이 종료되었습니다.")

    df_result = pd.DataFrame(session_results)
    if not append_each_streamer and not df_result.empty:
        df_result.to_csv(output_file, index=False, encoding="utf-8-sig")
    return df_result

### 3단계 : 소프트콘 데이터에서 SOOP 스트리머만 뽑고 이미 수집한 사람은 제외한 뒤 추가 수집

In [5]:
# ============================================================
def prepare_softcone_soop_targets(
    softcone_file: str = "softcone_streamer_dataset_clean.csv",
    existing_rank_file: str = "poongtoday_vtuber_rankings.csv",
    suffix: str = "softcone_clean",
) -> pd.DataFrame:
    """소프트콘 데이터에서 SOOP만 추출하고, 기존 풍투데이 랭킹 수집분/저장분을 제외한 타겟 생성."""
    df = pd.read_csv(softcone_file)
    df["channel_id"] = clean_channel_id(df["channel_id"])

    df_soop = df[df["platform"] == "SOOP"].copy()
    missing_mask = is_missing_channel_id(df_soop["channel_id"])

    df_missing = df_soop[missing_mask]
    missing_file = os.path.join(NULL_DIR, f"soop_missing_channel_id_{suffix}.csv")
    df_missing.to_csv(missing_file, index=False, encoding="utf-8-sig")

    df_valid = df_soop[~missing_mask].copy()
    # 방송 단위 데이터라면 streamer/channel이 중복되므로 채널 단위로 정리
    df_valid = df_valid.drop_duplicates("channel_id")

    existing_ids = set()
    candidate_existing_files = [
        existing_rank_file,
        os.path.join(DONE_DIR, f"new_vtuber_data_{suffix}.csv"),
        os.path.join(WRONG_DIR, f"wrong_channel_id_{suffix}.csv"),
    ]

    for path in candidate_existing_files:
        if os.path.exists(path):
            try:
                temp = pd.read_csv(path)
                if "channel_id" in temp.columns:
                    existing_ids.update(clean_channel_id(temp["channel_id"]).dropna().unique())
            except Exception:
                pass

    target_df = df_valid[~df_valid["channel_id"].isin(existing_ids)][["streamer_name", "channel_id"]].copy()
    target_df = target_df.rename(columns={"streamer_name": "name"})
    target_df["rank"] = pd.NA

    print("전체 스트리머 수:", df["streamer_name"].nunique())
    print("플랫폼별 스트리머 수:")
    print(df.groupby("platform")["streamer_name"].nunique())
    print("SOOP 전체 행 수:", len(df_soop))
    print("SOOP 스트리머 수:", df_soop["streamer_name"].nunique())
    print("고유 채널 수:", df_valid["channel_id"].nunique())
    print("channel_id 없는 스트리머 수:", df_soop.loc[missing_mask, "streamer_name"].nunique())
    print(f"결측 저장 완료: {missing_file}")
    print(f"기존/완료/실패 제외 후 새 타겟: {len(target_df)}명")

    return target_df


def crawl_softcone_soop_poong(
    softcone_file: str = "softcone_streamer_dataset_clean.csv",
    existing_rank_file: str = "poongtoday_vtuber_rankings.csv",
    suffix: str = "softcone_clean",
    headless: bool = False,
) -> pd.DataFrame:
    """소프트콘 SOOP 타겟의 월별 풍 데이터 수집."""
    target_df = prepare_softcone_soop_targets(
        softcone_file=softcone_file,
        existing_rank_file=existing_rank_file,
        suffix=suffix,
    )

    success_file = os.path.join(DONE_DIR, f"new_vtuber_data_{suffix}.csv")
    wrong_file = os.path.join(WRONG_DIR, f"wrong_channel_id_{suffix}.csv")

    return crawl_monthly_poong_by_ids(
        target_df=target_df,
        output_file=success_file,
        wrong_file=wrong_file,
        headless=headless,
        append_each_streamer=True,
    )

### 4단계 : 기존 풍투데이 수집분 + 소프트콘 추가 수집분을 합쳐서 최종 CSV 저장

In [6]:
# ============================================================
def combine_poong_files(
    poong_ranking_file: str = "poongtoday_vtuber_rankings.csv",
    softcone_done_file: str = os.path.join(DONE_DIR, "new_vtuber_data_softcone_clean.csv"),
    output_file: str = os.path.join(RESULT_DIR, "poongtoday_combined_data1.csv"),
) -> pd.DataFrame:
    """기존 풍투데이 랭킹 데이터와 소프트콘 추가 수집 데이터를 합쳐 최종 CSV 저장."""
    dfs = []

    if os.path.exists(poong_ranking_file):
        poong_df = pd.read_csv(poong_ranking_file)
        if "rank" in poong_df.columns:
            poong_df = poong_df.drop(columns=["rank"])
        dfs.append(poong_df)
    else:
        print(f"기존 풍투데이 랭킹 파일 없음: {poong_ranking_file}")

    if os.path.exists(softcone_done_file):
        softcone_df = pd.read_csv(softcone_done_file)
        if "rank" in softcone_df.columns:
            softcone_df = softcone_df.drop(columns=["rank"])
        dfs.append(softcone_df)
    else:
        print(f"소프트콘 추가 수집 파일 없음: {softcone_done_file}")

    if not dfs:
        raise FileNotFoundError("합칠 CSV가 없습니다. 파일 경로를 확인해주세요.")

    combined_df = pd.concat(dfs, ignore_index=True)

    # 컬럼 순서 통일
    wanted_cols = ["name", "channel_id", "year", "month", "total_star", "hourly_star"]
    existing_cols = [c for c in wanted_cols if c in combined_df.columns]
    other_cols = [c for c in combined_df.columns if c not in existing_cols]
    combined_df = combined_df[existing_cols + other_cols]

    # 타입 정리
    for col in ["year", "month", "total_star", "hourly_star"]:
        if col in combined_df.columns:
            combined_df[col] = pd.to_numeric(combined_df[col], errors="coerce").fillna(0).astype(int)

    # 같은 사람/같은 월 중복 제거
    before = len(combined_df)
    combined_df = combined_df.drop_duplicates(subset=["channel_id", "year", "month"], keep="first")
    duplicate_removed = before - len(combined_df)

    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    combined_df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f"통합 저장 완료: {output_file}")
    print(f"전체 행 수: {len(combined_df)}")
    print(f"고유 스트리머 수: {combined_df['channel_id'].nunique() if 'channel_id' in combined_df.columns else '확인 불가'}")
    print(f"제거된 중복 행 수: {duplicate_removed}")

    return combined_df

### 실행 예시

필요한 단계만 주석 해지해서 실행

In [ ]:
# ============================================================
if __name__ == "__main__":
    # 1) 풍투데이 버추얼 랭킹 ID 수집 예시
    #df_rank = crawl_virtual_ranking(start_rank=1, end_rank=100, output_file="poongtoday_vtuber_ranking_ids.csv")

    # 2) 수집한 랭킹 ID의 월별 풍 데이터 수집 예시
    # df_rank = pd.read_csv("poongtoday_vtuber_ranking_ids.csv")
    # crawl_monthly_poong_by_ids(
    #     target_df=df_rank,
    #     output_file="poongtoday_vtuber_rankings.csv",
    #     wrong_file="wrong_poongtoday_ranking_ids.csv",
    # )

    # 3) 소프트콘 SOOP 기준 추가 수집 예시
    # crawl_softcone_soop_poong(
    #     softcone_file="softcone_streamer_dataset_clean.csv",
    #     existing_rank_file="poongtoday_vtuber_rankings.csv",
    #     suffix="softcone_clean",
    # )

    # 4) 최종 통합 파일 생성 예시
    # combine_poong_files(
    #     poong_ranking_file="poongtoday_vtuber_rankings.csv",
    #     softcone_done_file="poong_save/done/new_vtuber_data_softcone_clean.csv",
    #     output_file="poong_save/result/poongtoday_combined_data1.csv",
    # )
    pass

전체 스트리머 수: 6641
플랫폼별 스트리머 수:
platform
CHZZK      4778
CIME        264
SOOP       1700
TWITCH        2
youtube       3
Name: streamer_name, dtype: int64
SOOP 전체 행 수: 444153
SOOP 스트리머 수: 1700
고유 채널 수: 1700
channel_id 없는 스트리머 수: 0
결측 저장 완료: poong_save\channel_id_null\soop_missing_channel_id_softcone_clean.csv
기존/완료/실패 제외 후 새 타겟: 0명
새롭게 수집할 타겟: 0명


### 얻을 수 있는 CSV

#### - poongtoday_vtuber_ranking_ids.csv : 풍투데이 랭킹 페이지에서 버추얼 스트리머의 이름, 순위, channel_id를 모은 파일


#### - poongtoday_vtuber_rankings.csv : 풍투데이 랭킹에서 가져온 스트리머들의 월별 별풍/시급풍 데이터

#### - poong_save/channel_id_null/soop_missing_channel_id_softcone_clean.csv : 소프트콘 SOOP 데이터 중 channel_id가 없는 사람만 따로 저장

#### - poong_save/done/new_vtuber_data_softcone_clean.csv : 소프트콘 SOOP 스트리머 중 풍투데이 랭킹 수집분에 없던 사람들을 추가로 수집한 결과

#### - poong_save/result/poongtoday_combined_data1.csv : 최종 취합 CSV

최종 취합 CSV 컬럼 : name, channel_id, year, month, total_star, hourly_star